# Exercício 14 — TutorGraph

**Entrega sem ecrã:** código executável abaixo + documentação em `docs/`.


## 0. Ambiente e caminhos

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

EX_ROOT = Path.cwd().resolve()
REPO_ROOT = EX_ROOT.parent.parent

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(EX_ROOT / ".env", override=True)

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no .env na raiz do repositório.")

print("OK — Repo:", REPO_ROOT)
print("OK — Exercício:", EX_ROOT)


## Fluxo tutor: diagnóstico → explicar → exercício → corrigir

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
import os

llm = ChatGoogleGenerativeAI(
    model=(os.environ.get("GEMINI_MODEL") or "gemini-2.0-flash").replace("models/", ""),
    temperature=0.3,
)

def step(system: str, human: str, **kw):
    p = ChatPromptTemplate.from_messages([("system", system), ("human", human)])
    return (p | llm | StrOutputParser()).invoke(kw)

tema = "Probabilidade condicionada"
diag = step("Diagnosticas lacunas.", "Aluno diz: confunde P(A|B) com P(B|A).", tema=tema)
exp = step("Explicas claro.", "Com base:\n{d}", d=diag)
ex = step("Propões exercício curto.", "Tema: {tema}", tema=tema)
sol_aluno = "0.3"
corr = step("Corriges.", "Enunciado:{ex}\nResposta aluno:{sol}", ex=ex, sol=sol_aluno)
print(diag, exp, ex, corr, sep="\n---\n")


## Referências
- `docs/` desta pasta — explicações detalhadas.
- `app/` — espelho API opcional.
